# 03a: Preprocessing, segmentation and quantification (batch)

**Step 1 of 3 of the analysis of one sample.** Runs unattended for all samples with `run_batch.py`, followed directly by **03b_cluster** (also automatic). Only **03c_annotate** needs you: subclustering and naming the clusters.

| Part | Step | Output (in `<OUTPUT_ROOT>/<sample>/`) |
|---|---|---|
| 0 | Sample parameters, marker panel, helpers | `<sample>_run_config.json` |
| 1 | Tissue/glass mask, **analysis region from your QuPath GeoJSON**, artefact mask, background subtraction | `<sample>_bgcorrected.ome.tif`, `<sample>_masks_lowres.npz` |
| 2 | Segmentation (Mesmer), per-cell quantification, quality filtering (cells outside the GeoJSON are removed) | `<sample>_nuclei_masks.tif`, `<sample>_wholecell_masks.tif` (only with membrane channels), `<sample>_masks_info.json` |
| 3 | Cell-level background correction | **`<sample>_cells_corrected.csv`** (input for 03b) |

Every figure carries the sample/condition label (e.g. `SS1_Primary_monophasic`) and is saved in **`<sample>/figures/`**, together with the figures of 03b and 03c.


---
# Part 0: Setup
## 0.1 Sample parameters
This cell is tagged `parameters`: when the notebook runs through `run_batch.py` (papermill), the values for each sample from `samples.csv` replace these defaults. List-type settings are written as comma-separated text, e.g. `'CD206'` or `'CD206,CD163'`.

| parameter | meaning |
|---|---|
| `FAILED_MARKERS` | antibodies that **did not stain in this sample**. Everything else in the panel is used for phenotyping; the failed ones are excluded and serve as the non-specific-signal reference. |
| `MEMBRANE_CHANNELS` | empty = nuclear segmentation (DAPI). With e.g. `CD206`, Mesmer segments nuclei **and** whole cells (DAPI + membrane channel). |
| `RING_MARKERS` | markers measured in the membrane/cytoplasm ring around the nucleus instead of in the nucleus (e.g. `CD206`). |
| `REGION_GEOJSON` | **required**: QuPath outline of the region to analyse (polygon tool → *Export objects as GeoJSON*). Only cells inside it are kept. |

In [ ]:
# ── sample (one row of samples.csv in batch mode) ─────────────────────────────────────────
SAMPLE            = 'ss3'                                                         # <-- EDIT
CONDITION         = 'primary_monophasic'                                          # <-- EDIT
REGION            = 'reg003'                                                      # <-- EDIT
RAW_IMAGE         = 'path/to/raw/ss3.ome.tif'        # <-- EDIT
CHANNEL_FILE      = 'path/to/raw/channelnames.txt'   # <-- EDIT
OUTPUT_ROOT       = 'path/to/output/'      # <-- EDIT one sub-folder per sample is created here

# ── sample-specific options (comma-separated text; empty = none) ──────────────────────────
FAILED_MARKERS    = 'EpCAM,FOXP3,b-catenin,Endomucin,VEGFA,CD206,TIE2,CD4,CD163,VEGFR1,PDGFRa,CTLA-4,CD45RO,CD14'   # antibodies that did not stain in this sample
MEMBRANE_CHANNELS = ''       # e.g. CD206 for nuclear + whole-cell segmentation
RING_MARKERS      = ''       # e.g. CD206 to measure it in the membrane ring
REGION_GEOJSON    = ''       # QuPath outline of the region to analyse (.geojson) - required

## 0.2 Paths, marker panel and file names
Derived from the parameters above. Do not edit this cell per sample; the full panel below lists the role of each antibody when it works; `FAILED_MARKERS` decides which of them are used in this sample.

In [ ]:
import os

def _as_list(s):
    """'CD206, CD163' -> ['CD206', 'CD163']; '' / nan -> []."""
    if isinstance(s, (list, tuple)):
        return [str(x).strip() for x in s if str(x).strip()]
    s = '' if s is None or str(s).lower() == 'nan' else str(s)
    return [x.strip() for x in s.split(',') if x.strip()]

FAILED_MARKERS, MEMBRANE_CHANNELS, RING_MARKERS = map(_as_list, (FAILED_MARKERS, MEMBRANE_CHANNELS, RING_MARKERS))
REGION_GEOJSON = '' if str(REGION_GEOJSON).lower() in ('', 'nan', 'none') else str(REGION_GEOJSON).strip()
assert REGION_GEOJSON, 'REGION_GEOJSON is empty: give the QuPath outline of this sample (column geojson in samples.csv)'
assert os.path.exists(REGION_GEOJSON), f'GeoJSON not found: {REGION_GEOJSON}'

NUCLEUS_CHANNEL = 'DAPI'
output_dir = os.path.join(OUTPUT_ROOT, SAMPLE) + os.sep            # per-sample folder
PLOT_TAG   = f'{SAMPLE.upper()}_{CONDITION[:1].upper()}{CONDITION[1:]}'   # e.g. SS3_Primary_monophasic

# ── full marker panel: the role each antibody has when it works ───────────────────────────
ALL_LINEAGE = ['CD45', 'CD3E', 'CD8', 'CD20', 'CD68',              # immune cells
               'CD31', 'VWF', 'CD34', 'aSMA', 'PDGFRb',            # vessels and stroma
               'col1A1', 'PanCK',                                  # matrix and epithelium
               'EpCAM', 'b-catenin', 'Endomucin', 'CD206', 'TIE2', 'VEGFR1', 'PDGFRa',
               'CD4', 'FOXP3', 'CD163', 'CD45RO', 'CD14']
ALL_STATE   = ['Ki67', 'HIF1A', 'VEGFA', 'CTLA-4']                 # proliferation, hypoxia, function
NUCLEAR_MARKERS = ['Ki67', 'HIF1A', 'FOXP3']                       # measured in the nucleus
# antibodies without a specific target in this tissue: when they fail, they are a good
# reference for non-specific binding / autofluorescence (Part 3.2)
REF_CANDIDATES = ['EpCAM', 'FOXP3', 'b-catenin', 'Endomucin', 'VEGFA', 'TIE2', 'VEGFR1',
                  'PDGFRa', 'CTLA-4']

# ── sample-specific panel, derived from the failed markers ────────────────────────────────
unknown = [m for m in FAILED_MARKERS if m not in ALL_LINEAGE + ALL_STATE]
assert not unknown, f'FAILED_MARKERS not in the panel (typo?): {unknown}'
LINEAGE_MARKERS = [m for m in ALL_LINEAGE if m not in FAILED_MARKERS]
STATE_MARKERS   = [m for m in ALL_STATE if m not in FAILED_MARKERS]
NONFUNCTIONAL   = list(FAILED_MARKERS)
REF_CHANNELS    = [m for m in REF_CANDIDATES if m in FAILED_MARKERS] or list(FAILED_MARKERS)
assert REF_CHANNELS, 'FAILED_MARKERS is empty: at least one non-binding antibody is needed as reference'

# ── output files ──────────────────────────────────────────────────────────────────────────
OUT            = output_dir + SAMPLE + '_'
OUT_IMAGE      = OUT + 'bgcorrected.ome.tif'
OUT_ARTEFACT   = OUT + 'artefact_mask.tif'
MASK_FILE      = OUT + 'nuclei_masks.tif'
CELL_MASK_FILE = OUT + 'wholecell_masks.tif'       # only with MEMBRANE_CHANNELS
CORRECTED_CSV  = OUT + 'cells_corrected.csv'       # input for 03b
RUN_CONFIG     = OUT + 'run_config.json'           # sample settings read by 03b
LOWRES_MASKS   = OUT + 'masks_lowres.npz'          # tissue and region masks read by 03b
FIG_DIR        = output_dir + 'figures/'
os.makedirs(FIG_DIR, exist_ok=True)

print(f'{PLOT_TAG}: output -> {output_dir}')
print('failed in this sample:', FAILED_MARKERS or '-',
      '| membrane channels:', MEMBRANE_CHANNELS or '-',
      '| ring markers:', RING_MARKERS or '-', '| region outline:', REGION_GEOJSON)
print('lineage markers:', LINEAGE_MARKERS)
print('state markers:', STATE_MARKERS, '| reference channels:', REF_CHANNELS)


## 0.3 Libraries, image information and figure helper
Loads the libraries, reads the channel names and the image size, and checks that every marker in the panel exists in the image. `finish()` is used for every figure: it adds the sample/condition label as a title and saves the figure as `figures/<PLOT_TAG>_<name>.png`.

In [ ]:
import os, gc, json, warnings
import numpy as np
import pandas as pd
import tifffile
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
from scipy.stats import theilslopes, spearmanr
from skimage.filters import threshold_otsu
from skimage.morphology import disk, remove_small_objects, remove_small_holes, binary_closing
from skimage.segmentation import expand_labels, find_boundaries
from skimage.measure import regionprops_table

warnings.filterwarnings('ignore')
plt.rc('axes', grid=False)
os.makedirs(output_dir, exist_ok=True)
PARAM_LOG = {'sample': SAMPLE, 'condition': CONDITION, 'region': REGION, 'failed_markers': FAILED_MARKERS,
             'membrane_channels': MEMBRANE_CHANNELS, 'ring_markers': RING_MARKERS}   # written to <sample>_parameters.json

# ── figure helper: sample/condition label on every figure + save ──────────────────────────
def finish(name, title=None, fig=None, dpi=300, tight=True):
    """Add '<PLOT_TAG> | [title]' above the figure, save it as figures/<PLOT_TAG>_<name>.png, show it."""
    fig = fig or plt.gcf()
    label = PLOT_TAG + (f'  |  {title}' if title else '')
    if fig.get_constrained_layout():                     # layout engine places the title itself
        fig.suptitle(label, fontsize=14, fontweight='bold')
    else:                                                # place it just above the axes
        fig.suptitle(label, fontsize=14, fontweight='bold', y=1.02)
    fig.savefig(FIG_DIR + f'{PLOT_TAG}_{name}.png', dpi=dpi, **(dict(bbox_inches='tight') if tight else {}))
    plt.show()

# ── channels and image size ───────────────────────────────────────────────────────────────
channel_names = [l.strip() for l in open(CHANNEL_FILE) if l.strip()]

def _channel_pages(tif, n):
    """Return the n channel pages (works for plain multipage and pyramidal OME-TIFFs)."""
    for cand in (list(tif.pages), [p for s in tif.series for p in s.pages],
                 list(tif.series[0].pages) if tif.series else []):
        if len(cand) == n:
            return cand
    raise AssertionError(f'cannot find {n} channel pages (tif.pages={len(tif.pages)})')

def read_plane(name, path=RAW_IMAGE):
    """Read one channel of an image as a 2D array."""
    with tifffile.TiffFile(path) as tif:
        return _channel_pages(tif, len(channel_names))[channel_names.index(name)].asarray()

with tifffile.TiffFile(RAW_IMAGE) as tif:
    _p = _channel_pages(tif, len(channel_names))
    H, W = _p[0].shape
    DTYPE = _p[0].dtype
MAXVAL = np.iinfo(DTYPE).max

# ── check the panel against the channel list ─────────────────────────────────────────────
marker_channels = [c for c in channel_names if c != NUCLEUS_CHANNEL]
for lst, nm in [(NONFUNCTIONAL, 'NONFUNCTIONAL'), (LINEAGE_MARKERS, 'LINEAGE_MARKERS'),
                (STATE_MARKERS, 'STATE_MARKERS'), (REF_CHANNELS, 'REF_CHANNELS')]:
    miss = [c for c in lst if c not in channel_names]
    assert not miss, f'{nm}: not in channelnames.txt: {miss}'
assert not set(REF_CHANNELS) & set(LINEAGE_MARKERS + STATE_MARKERS), 'a reference channel is also a phenotyping marker'
PHENO_MARKERS = LINEAGE_MARKERS + STATE_MARKERS

print(f'{PLOT_TAG}: {len(channel_names)} channels, {H} x {W} px, {DTYPE}')
print(f'phenotyping markers ({len(PHENO_MARKERS)}): {PHENO_MARKERS}')
print(f'reference channels  ({len(REF_CHANNELS)}): {REF_CHANNELS}')

## 0.4 Helper functions for background estimation
Background is estimated on a down-sampled copy of each channel (4 × 4 pixels averaged into one, `BG_DS = 4`) and interpolated back to full resolution. The background varies slowly across the slide, so no detail is lost, and a 10 000 × 15 000 px image stays fast to process.

In [ ]:
BG_DS = 4      # down-sampling factor for background estimation

def pool(img, f=BG_DS):
    """Average f x f pixel blocks (down-sampling)."""
    h, w = img.shape[0] // f, img.shape[1] // f
    return img[:h * f, :w * f].astype(np.float32).reshape(h, f, w, f).mean(axis=(1, 3))

def pool_any(mask, f=BG_DS):
    """Down-sample a binary mask: a block is True if any pixel in it is True."""
    h, w = mask.shape[0] // f, mask.shape[1] // f
    return mask[:h * f, :w * f].reshape(h, f, w, f).any(axis=(1, 3))

def _interp_axis(n_out, n_in, block):
    c = np.clip((np.arange(n_out) + 0.5) / block - 0.5, 0, n_in - 1)
    i0 = np.floor(c).astype(np.int64)
    i1 = np.minimum(i0 + 1, n_in - 1)
    return i0, i1, (c - i0).astype(np.float32)

def upsample_grid(A, block, shape):
    """Bilinear interpolation of a coarse grid (cells `block` px wide) back to full resolution."""
    A = A.astype(np.float32)
    y0, y1, wy = _interp_axis(shape[0], A.shape[0], block)
    x0, x1, wx = _interp_axis(shape[1], A.shape[1], block)
    Ax = A[:, x0] * (1 - wx) + A[:, x1] * wx
    out = Ax[y0]; out *= (1 - wy)[:, None]
    tmp = Ax[y1]; tmp *= wy[:, None]; out += tmp
    return out

def block_view(a, b):
    nby, nbx = -(-a.shape[0] // b), -(-a.shape[1] // b)
    pad = np.full((nby * b, nbx * b), np.nan, np.float32)
    pad[:a.shape[0], :a.shape[1]] = a
    return pad.reshape(nby, b, nbx, b).transpose(0, 2, 1, 3).reshape(nby, nbx, b * b)

def block_quantile(small, valid, b, q, min_frac=0.10):
    """q-th percentile of the valid (tissue) pixels in each b x b block."""
    v = block_view(np.where(valid, small, np.nan), b)
    frac = np.isfinite(v).mean(axis=2)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        out = np.nanpercentile(v, q, axis=2)
    out[frac < min_frac] = np.nan
    return out

def fill_smooth(grid, sigma):
    """Fill empty grid cells from their neighbours and smooth the grid (normalised Gaussian)."""
    w = np.isfinite(grid).astype(np.float32)
    if w.sum() == 0:
        return np.zeros_like(grid, dtype=np.float32)
    g = np.nan_to_num(grid).astype(np.float32)
    out = np.full(g.shape, np.nan, np.float32)
    for s in (max(sigma, 0.5), 4 * max(sigma, 0.5) + 4, 50.0):
        num = ndi.gaussian_filter(g * w, s, mode='nearest')
        den = ndi.gaussian_filter(w, s, mode='nearest')
        est = num / np.maximum(den, 1e-6)
        todo = np.isnan(out) & (den > 1e-3)
        out[todo] = est[todo]
    out[np.isnan(out)] = np.nanmedian(grid)
    return out

def robust_sd(x):
    """Standard deviation estimated from the median absolute deviation (insensitive to outliers)."""
    x = np.asarray(x, dtype=np.float64); x = x[np.isfinite(x)]
    return 1.4826 * np.median(np.abs(x - np.median(x))) if x.size else np.nan

def show_lowres(a, ax, title, **kw):
    ax.imshow(a, interpolation='nearest', **kw); ax.set_title(title, fontsize=9); ax.axis('off')

---
# Part 1: Pixel-level preprocessing
## 1.1 Tissue/glass mask
Separates tissue from the empty glass around it, from DAPI (smoothed, log-transformed, Otsu threshold; small holes filled, small fragments removed). It is **not** the region that is analysed; that comes from your GeoJSON (1.1b). This mask is only used for two technical steps:
* background statistics are computed on tissue pixels, not on glass (1.3);
* cells right at the physical edge of the section are removed, because staining there is unreliable (2.4).


In [ ]:
TISSUE_SMOOTH      = 4        # <-- EDIT Gaussian smoothing of DAPI (in down-sampled px)
TISSUE_OTSU_SCALE  = 1.0      # <-- EDIT < 1 makes the mask more generous
TISSUE_MAX_HOLE_PX = 40_000   # <-- EDIT holes smaller than this (px²) are filled
TISSUE_MIN_OBJ_PX  = 200_000  # <-- EDIT tissue pieces smaller than this (px²) are removed

dapi_lr = pool(read_plane(NUCLEUS_CHANNEL))                       # down-sampled DAPI
lg = ndi.gaussian_filter(np.log1p(dapi_lr), TISSUE_SMOOTH)        # smoothed log-DAPI
t_otsu = threshold_otsu(lg[::2, ::2]) * TISSUE_OTSU_SCALE         # automatic threshold
tissue_lr = binary_closing(lg > t_otsu, disk(3))
tissue_lr = remove_small_holes(tissue_lr, TISSUE_MAX_HOLE_PX // BG_DS ** 2)
tissue_lr = remove_small_objects(tissue_lr, TISSUE_MIN_OBJ_PX // BG_DS ** 2)
print(f'tissue: {100 * tissue_lr.mean():.1f} % of the image')

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
show_lowres(dapi_lr, ax[0], 'DAPI', cmap='gray', vmax=np.percentile(dapi_lr, 99.5))
ax[1].imshow(dapi_lr, cmap='gray', vmax=np.percentile(dapi_lr, 99.5))
ax[1].contour(tissue_lr, [0.5], colors='yellow', linewidths=0.8)
ax[1].set_title('tissue mask (yellow outline)'); ax[1].axis('off')
plt.tight_layout(); finish('1-01_tissue_mask', 'Tissue mask', dpi=150)

## 1.1b Analysis region (QuPath GeoJSON)
The region to analyse is read from your QuPath outline (`REGION_GEOJSON`). Holes in the outline are respected. Segmentation and background correction still run on the whole image, but only cells whose centre lies inside the outline are kept (2.4). The red outline in the figure must match what you drew in QuPath.


In [ ]:
from skimage.draw import polygon as draw_polygon
from matplotlib.path import Path as MplPath

def _rings(geom):
    """Yield (ring_coordinates, is_outline) for any GeoJSON geometry type."""
    if not geom:
        return
    t, c = geom.get('type'), geom.get('coordinates')
    if t == 'Polygon':
        for k, ring in enumerate(c):
            yield ring, k == 0
    elif t == 'MultiPolygon':
        for poly in c:
            for k, ring in enumerate(poly):
                yield ring, k == 0
    elif t in ('LineString', 'MultiLineString'):
        for line in ([c] if t == 'LineString' else c):
            if len(line) >= 3:
                yield line, True
    elif t == 'GeometryCollection':
        for g in geom.get('geometries', []):
            yield from _rings(g)

def geojson_rings(path):
    """[(ring as (n, 2) array of x, y in full-resolution px, is_outline)] for all objects in a QuPath GeoJSON."""
    gj = json.load(open(path))
    feats = gj['features'] if isinstance(gj, dict) and 'features' in gj else (gj if isinstance(gj, list) else [gj])
    out = []
    for f in feats:
        geom = f.get('geometry', f) if isinstance(f, dict) else None
        out += [(np.asarray(r, float).reshape(-1, 2), o) for r, o in _rings(geom)]
    assert any(o for _, o in out), f'no area found in {path}'
    return out

def mask_from_geojson(path, shape, ds=4):
    """Rasterise the outline onto a grid down-sampled by ds (outlines add, holes remove)."""
    m = np.zeros(shape, bool)
    for r, outline in geojson_rings(path):
        rr, cc = draw_polygon(r[:, 1] / ds, r[:, 0] / ds, shape)
        m[rr, cc] = outline
    return m

def points_in_geojson(path, x, y):
    """True for every point (full-resolution px) inside the outline (outlines add, holes remove)."""
    pts = np.column_stack([x, y]); inside = np.zeros(len(pts), bool)
    for r, outline in geojson_rings(path):
        hit = MplPath(r).contains_points(pts)
        inside = (inside | hit) if outline else (inside & ~hit)
    return inside

outline_lr = mask_from_geojson(REGION_GEOJSON, tissue_lr.shape, BG_DS)
region_lr = outline_lr & tissue_lr                   # region without empty glass (used for its area)
print(f'analysis region: {100 * outline_lr.mean():.1f} % of the image, {100 * region_lr.sum() / tissue_lr.sum():.1f} % of the tissue')

fig, ax = plt.subplots(figsize=(8, 8 * tissue_lr.shape[0] / tissue_lr.shape[1]))
ax.imshow(dapi_lr, cmap='gray', vmax=np.percentile(dapi_lr, 99.5))
ax.contour(tissue_lr, [0.5], colors='yellow', linewidths=0.6)
ax.contour(outline_lr, [0.5], colors='red', linewidths=1.2)
ax.set_title('tissue (yellow) and analysis region from the GeoJSON (red)'); ax.axis('off')
finish('1-01b_analysis_region', 'Analysis region (QuPath GeoJSON)', dpi=150)


## 1.2 Artefact mask (folds, debris, dust)
Artefacts such as tissue folds and debris are bright in almost every channel at once, whereas real cells are bright in only a few markers. For each channel, a pixel counts as *positive* if it lies more than `ARTIFACT_NMAD` robust standard deviations above the channel median. A pixel positive in more than `ARTIFACT_FRAC` of the 28 marker channels is marked as artefact. Isolated specks are ignored, and artefact regions are slightly enlarged to also cover their bright rim.

In [ ]:
ARTIFACT_NMAD      = 5.0     # <-- EDIT "positive" = above median + 5 robust SD
ARTIFACT_FRAC      = 0.50    # <-- EDIT artefact = positive in > 50 % of marker channels
ARTIFACT_MIN_PX    = 50      # <-- EDIT ignore specks smaller than this (px)
ARTIFACT_DILATE_PX = 8       # <-- EDIT enlarge artefact regions by this many px
PCT_SUBSAMPLE      = 5_000_000

# count, per pixel, in how many marker channels it is positive
rng = np.random.default_rng(0)
counter = np.zeros((H, W), dtype=np.uint8)
with tifffile.TiffFile(RAW_IMAGE) as tif:
    pages = _channel_pages(tif, len(channel_names))
    for name, page in zip(channel_names, pages):
        if name == NUCLEUS_CHANNEL:
            continue
        img = page.asarray()
        flat = img.ravel()
        if flat.size > PCT_SUBSAMPLE:                       # subsample for speed
            flat = rng.choice(flat, PCT_SUBSAMPLE, replace=False)
        med = np.median(flat)
        mad = np.median(np.abs(flat.astype(np.float64) - med)) * 1.4826
        counter += (img > med + ARTIFACT_NMAD * max(mad, 1.0)).astype(np.uint8)
        del img

# artefact = positive in too many channels; remove specks, then enlarge each region
n_required = int(np.floor(ARTIFACT_FRAC * len(marker_channels))) + 1
artifact = remove_small_objects(counter >= n_required, ARTIFACT_MIN_PX)
if ARTIFACT_DILATE_PX > 0 and artifact.any():
    lbl, nb = ndi.label(artifact)
    se = disk(ARTIFACT_DILATE_PX).astype(bool)
    r = ARTIFACT_DILATE_PX
    for sl in ndi.find_objects(lbl):                       # dilate each region in its bounding box
        y0, y1 = max(sl[0].start - r, 0), min(sl[0].stop + r, H)
        x0, x1 = max(sl[1].start - r, 0), min(sl[1].stop + r, W)
        artifact[y0:y1, x0:x1] |= ndi.binary_dilation(artifact[y0:y1, x0:x1], se)
art_lr = pool_any(artifact)
counts_px = np.bincount(counter.ravel(), minlength=len(marker_channels) + 1)
del counter; gc.collect()
print(f'artefact = positive in >= {n_required} of {len(marker_channels)} marker channels')
print(f'artefact pixels: {artifact.sum():,} ({100 * artifact.mean():.4f} % of the image)')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].bar(np.arange(len(counts_px)), counts_px, color='0.6'); ax[0].set_yscale('log')
ax[0].axvline(n_required - 0.5, color='tab:red', lw=2, label=f'artefact cut-off: >= {n_required}')
ax[0].set_xlabel('number of marker channels in which the pixel is positive'); ax[0].set_ylabel('pixels'); ax[0].legend()
ax[1].imshow(dapi_lr, cmap='gray', vmax=np.percentile(dapi_lr, 99.5))
ax[1].imshow(np.ma.masked_where(~art_lr, art_lr), cmap='autumn', interpolation='nearest')
ax[1].set_title('artefacts (red) on DAPI'); ax[1].axis('off')
plt.tight_layout(); finish('1-02_artefact_mask', 'Artefact detection', dpi=150)
tifffile.imwrite(OUT_ARTEFACT, (artifact * 255).astype(np.uint8), compression='zlib')

## 1.3 Background subtraction: methods and settings
Each marker channel gets a **background surface**, a smooth estimate of the signal that is not specific staining. The surface is subtracted from the image, and negative values are set to 0. The default method, `quantile_surface`, divides the tissue into blocks of `block` × `block` px and takes a low percentile (`q`) of the tissue pixels in each block as the local background. The blocks are then smoothed into a continuous surface. This follows regional background differences and tile-to-tile intensity steps of the scanner.

| method | background estimate | use |
|---|---|---|
| `quantile_surface` | low percentile per block, smoothed | default for all markers |
| `opening` | morphological opening (rolling-ball-like) | sparse markers on a smooth background |
| `local_mean` | `k` × local mean intensity | alternative |
| `none` | no subtraction | DAPI (needed raw for segmentation) |

Markers that cover large continuous areas (collagen, dense immune aggregates) use larger blocks and a lower percentile, so their real signal is not mistaken for background.

In [ ]:
BG_DEFAULT = dict(method='quantile_surface', block=256, q=20, smooth=1.0, radius=100, k=1.5)
BG_PARAMS = {                                     # <-- EDIT per-channel settings
    NUCLEUS_CHANNEL: dict(method='none'),         # DAPI stays raw for segmentation
    'col1A1':        dict(block=512, q=10),       # collagen covers large areas
    'CD45':          dict(block=384, q=15),       # dense immune aggregates
    'PDGFRb':        dict(block=384, q=15),
}
FLATFIELD_TILE = None      # <-- EDIT optional: tile size (px) for flat-field correction; None = off
NOISE_NMAD     = 0.0       # <-- EDIT optional extra noise floor (robust SDs); 0 = off

def bg_params(name):
    """Settings for one channel = defaults + channel-specific overrides."""
    p = dict(BG_DEFAULT); p.update(BG_PARAMS.get(name, {})); return p

valid_lr = tissue_lr & ~art_lr      # tissue pixels without artefacts

def flatfield_profile(img, tile):
    """Median intensity profile across acquisition tiles (for optional flat-field correction)."""
    th, tw = tile[0] // BG_DS, tile[1] // BG_DS
    small = np.where(valid_lr, pool(img), np.nan)
    ny, nx = small.shape[0] // th, small.shape[1] // tw
    stack = small[:ny * th, :nx * tw].reshape(ny, th, nx, tw).transpose(0, 2, 1, 3).reshape(-1, th, tw)
    prof = fill_smooth(np.nanmedian(stack, axis=0), sigma=max(th, tw) / 16)
    prof /= np.nanmean(prof)
    return upsample_grid(prof, BG_DS, tile)

def apply_flatfield(img, tile):
    prof = flatfield_profile(img, tile)
    reps = (-(-img.shape[0] // tile[0]), -(-img.shape[1] // tile[1]))
    img /= np.tile(prof, reps)[:img.shape[0], :img.shape[1]]
    return img

def estimate_background(small, p):
    """Background surface of one down-sampled channel -> (grid, grid cell size in full-res px)."""
    m = p['method']
    if m == 'none':
        return None, None
    if m == 'quantile_surface':
        b = max(1, p['block'] // BG_DS)
        return fill_smooth(block_quantile(small, valid_lr, b, p['q']), p['smooth']), b * BG_DS
    if m == 'opening':
        r = max(1, p['radius'] // BG_DS)
        a = np.where(art_lr, np.median(small[valid_lr]), small)
        bg = ndi.maximum_filter(ndi.minimum_filter(a, size=2 * r + 1), size=2 * r + 1)
        return ndi.gaussian_filter(bg, r / 2), BG_DS
    if m == 'local_mean':
        r = max(1, p['radius'] // BG_DS); v = valid_lr.astype(np.float32)
        num = ndi.uniform_filter(small * v, 2 * r + 1); den = ndi.uniform_filter(v, 2 * r + 1)
        return p['k'] * num / np.maximum(den, 1e-6), BG_DS
    raise ValueError(f'unknown background method {m!r}')

def correct_plane(name, raw):
    """Background-correct one channel: subtract the surface, clip at 0, zero the artefacts."""
    p = bg_params(name)
    img = raw.astype(np.float32)
    if FLATFIELD_TILE and p['method'] != 'none':
        img = apply_flatfield(img, FLATFIELD_TILE)
    small = pool(img)
    grid, blk = estimate_background(small, p)
    if grid is not None:
        if NOISE_NMAD > 0:
            res = small - upsample_grid(grid, blk // BG_DS, small.shape)
            grid = grid + NOISE_NMAD * robust_sd(res[valid_lr])
        img -= upsample_grid(grid, blk, img.shape)
        np.clip(img, 0, None, out=img)
    img[artifact] = 0
    return img, grid, blk

print('background settings per channel:')
print(pd.DataFrame({c: bg_params(c) for c in channel_names}).T[['method', 'block', 'q']].to_string())

## 1.4 Preview of the background correction
Before all channels are processed, the correction is shown for a few markers on a 600 × 600 px crop from the tissue centre:
* **Columns 1–3**: raw image, estimated background, corrected image.
* **Column 4**: the background surface across the whole slide.
* **Columns 5–6**: the median intensity per 500 px block inside the tissue, before and after correction. Before, regional differences and tile steps are visible. After, the map should be uniform.

The correction is right when diffuse background disappears while positive cells keep their shape and brightness.

In [ ]:
PREVIEW_CHANNELS = ['CD45', 'CD3E', 'CD68', 'col1A1']    # <-- EDIT
PREVIEW_CROP     = None    # <-- EDIT (y0, y1, x0, x1) in px; None = crop at the tissue centre
TILE_QC_PX       = 500     # block size for the uniformity maps

if PREVIEW_CROP is None:
    cy, cx = [int(v * BG_DS) for v in ndi.center_of_mass(region_lr if region_lr.any() else tissue_lr)]
    PREVIEW_CROP = (max(cy - 300, 0), min(cy + 300, H), max(cx - 300, 0), min(cx + 300, W))
y0, y1, x0, x1 = PREVIEW_CROP

def tissue_block_median(small):
    """Median tissue intensity per 500 px block (uniformity check)."""
    return block_quantile(small, valid_lr, TILE_QC_PX // BG_DS, 50, min_frac=0.3)

chans = [c for c in PREVIEW_CHANNELS if c in channel_names]
fig, axes = plt.subplots(len(chans), 6, figsize=(24, 4 * len(chans)), squeeze=False)
for r, name in enumerate(chans):
    raw = read_plane(name)
    cor, grid, blk = correct_plane(name, raw)
    bg_crop = upsample_grid(grid, blk, (H, W))[y0:y1, x0:x1] if grid is not None else np.zeros((y1 - y0, x1 - x0))
    a, c = raw[y0:y1, x0:x1].astype(np.float32), cor[y0:y1, x0:x1]
    vmax = np.percentile(a, 99.5)
    axes[r, 0].imshow(a, cmap='magma', vmin=0, vmax=vmax); axes[r, 0].set_title(f'{name} raw')
    axes[r, 1].imshow(bg_crop, cmap='magma', vmin=0, vmax=vmax); axes[r, 1].set_title('estimated background')
    axes[r, 2].imshow(c, cmap='magma', vmin=0, vmax=max(np.percentile(c, 99.5), 1))
    axes[r, 2].set_title(f'{name} corrected')
    if grid is not None:
        show_lowres(grid, axes[r, 3], 'background surface (whole slide)', cmap='magma')
    b_raw, b_cor = tissue_block_median(pool(raw)), tissue_block_median(pool(cor))
    d_raw, d_cor = b_raw - np.nanmedian(b_raw), b_cor - np.nanmedian(b_cor)
    lim = np.nanpercentile(np.abs(d_raw), 95)          # same colour scale for both maps
    show_lowres(d_raw, axes[r, 4], f'block median, raw (sd {np.nanstd(b_raw):.0f})', cmap='RdBu_r', vmin=-lim, vmax=lim)
    show_lowres(d_cor, axes[r, 5], f'block median, corrected (sd {np.nanstd(b_cor):.0f})', cmap='RdBu_r', vmin=-lim, vmax=lim)
    for k in range(3):
        axes[r, k].axis('off')
    del raw, cor; gc.collect()
plt.tight_layout(); finish('1-04_background_preview', 'Background correction: preview', dpi=150)

## 1.5 Background-correct all channels and save the image
All channels are corrected with the settings from 1.3 and written to a new OME-TIFF. DAPI is kept raw, and only artefact pixels are set to 0. The file is pyramidal (full resolution, 1/4, 1/16) with channel names and colours, so it opens smoothly in QuPath. The printed table reports, per channel, how much the regional background variation (`block_sd`) was reduced.

In [ ]:
TILE    = (1024, 1024)       # TIFF tile size
PYRAMID = [4, 16]            # extra resolution levels for QuPath
PALETTE = [(255, 0, 0), (0, 255, 0), (255, 0, 255), (255, 255, 0), (0, 255, 255), (255, 128, 0),
           (128, 0, 255), (0, 128, 255), (255, 0, 128), (128, 255, 0), (0, 255, 128), (255, 128, 128),
           (128, 128, 255), (255, 200, 0), (0, 200, 200), (200, 0, 200), (150, 255, 50), (255, 80, 80),
           (80, 255, 255), (255, 50, 200), (50, 200, 255), (200, 255, 50), (255, 150, 50), (150, 50, 255),
           (50, 255, 150), (255, 255, 128), (128, 255, 255), (255, 128, 255)]
CHANNEL_COLORS = {NUCLEUS_CHANNEL: (0, 0, 255)}       # <-- EDIT display colours in QuPath

def _rgba(rgb):
    return int.from_bytes(bytes([*rgb, 255]), 'big', signed=True)
_i, colors = 0, []
for c in channel_names:
    if c in CHANNEL_COLORS:
        colors.append(_rgba(CHANNEL_COLORS[c]))
    else:
        colors.append(_rgba(PALETTE[_i % len(PALETTE)])); _i += 1

def _downsample(plane, f):
    h, w = plane.shape[0] // f, plane.shape[1] // f
    return plane[:h * f, :w * f].astype(np.float32).reshape(h, f, w, f).mean(axis=(1, 3)).astype(plane.dtype)

_levels = {f: [] for f in PYRAMID}
stats = []
_ty, _tx = np.nonzero(valid_lr[::2, ::2]); _ty, _tx = _ty * 2 * BG_DS, _tx * 2 * BG_DS   # sample of tissue pixels

def _planes():
    """Read, correct and yield one channel at a time (keeps memory low)."""
    with tifffile.TiffFile(RAW_IMAGE) as tif:
        pages = _channel_pages(tif, len(channel_names))
        for name, page in zip(channel_names, pages):
            raw = page.asarray()
            cor, grid, blk = correct_plane(name, raw)
            b_raw, b_cor = tissue_block_median(pool(raw)), tissue_block_median(pool(cor))
            stats.append(dict(channel=name, method=bg_params(name)['method'],
                              raw_tissue_median=float(np.median(raw[_ty, _tx])),
                              bg_mean=float(np.nanmean(grid)) if grid is not None else 0.0,
                              block_sd_raw=float(np.nanstd(b_raw)), block_sd_corrected=float(np.nanstd(b_cor)),
                              pct_px_nonzero=round(100 * float((cor[_ty, _tx] > 0).mean()), 1)))
            plane = np.clip(cor, 0, MAXVAL).astype(DTYPE)
            del raw, cor
            for f in PYRAMID:
                _levels[f].append(_downsample(plane, f))
            yield plane

def _tiles():
    th, tw = TILE
    for plane in _planes():
        for y in range(0, H, th):
            for x in range(0, W, tw):
                yield plane[y:y + th, x:x + tw]

# write the pyramidal OME-TIFF
opts = dict(photometric='minisblack', tile=TILE, compression='zlib')
with tifffile.TiffWriter(OUT_IMAGE, bigtiff=True) as w:
    w.write(_tiles(), shape=(len(channel_names), H, W), dtype=DTYPE, subifds=len(PYRAMID),
            metadata={'axes': 'CYX', 'Channel': {'Name': channel_names, 'Color': colors}}, **opts)
    for f in PYRAMID:
        w.write(np.stack(_levels.pop(f)), subfiletype=1, **opts)

bg_log = pd.DataFrame(stats).set_index('channel')
bg_log['block_sd_reduction_%'] = (100 * (1 - bg_log.block_sd_corrected / bg_log.block_sd_raw)).round(0)
bg_log.to_csv(OUT + 'bg_correction_log.csv')
PARAM_LOG['pixel_background'] = {c: bg_params(c) for c in channel_names}
PARAM_LOG['flatfield_tile'] = FLATFIELD_TILE; PARAM_LOG['noise_nmad'] = NOISE_NMAD
print(bg_log.round(1).to_string())
print(f'\nwritten: {OUT_IMAGE} ({os.path.getsize(OUT_IMAGE) / 1e9:.2f} GB)')

---
# Part 2: Segmentation and quantification
## 2.1 Segmentation (Mesmer)
Nuclei are segmented with **Mesmer** (DeepCell) through SPACEc, on the DAPI channel of the background-corrected image. If `MEMBRANE_CHANNELS` is set, Mesmer additionally segments whole cells from DAPI plus the membrane channel(s) (`differentiate_nucleus_cytoplasm=True`). The two masks are numbered independently, so each nucleus is given the ID of the cell it lies in. This is the slowest step of the pipeline.

**Re-runs.** The masks are saved together with the settings they were made with (`<sample>_masks_info.json`: membrane channels, nuclear channel, method, raw image). On a re-run they are reused only if these settings are unchanged; if you change `membrane_channels` in samples.csv (adding, removing or emptying it), the segmentation runs again automatically and the old masks are replaced. Changing `ring_markers` or `failed_markers` does not need new masks. Set `REUSE_MASKS = False` to force a new segmentation.


In [ ]:
SEG_METHOD    = 'mesmer'
RESIZE_FACTOR = 1
REUSE_MASKS   = True          # reuse saved masks if they were made with the same settings (False = always segment)
WHOLE_CELL    = len(MEMBRANE_CHANNELS) > 0
MASK_INFO     = OUT + 'masks_info.json'          # settings the saved masks were made with

seg_settings = dict(SEG_METHOD=SEG_METHOD, NUCLEUS_CHANNEL=NUCLEUS_CHANNEL, RESIZE_FACTOR=RESIZE_FACTOR,
                    MEMBRANE_CHANNELS=sorted(MEMBRANE_CHANNELS),
                    RAW_IMAGE=os.path.basename(RAW_IMAGE), RAW_IMAGE_BYTES=os.path.getsize(RAW_IMAGE))

def saved_mask_settings():
    """(settings, source) of the masks on disk, or (None, reason)."""
    if os.path.exists(MASK_INFO):
        return json.load(open(MASK_INFO)), 'masks_info.json'
    if os.path.exists(RUN_CONFIG):           # masks from before this check existed: the previous run's settings
        cfg = json.load(open(RUN_CONFIG))
        if 'MEMBRANE_CHANNELS' in cfg:
            return dict(MEMBRANE_CHANNELS=sorted(cfg['MEMBRANE_CHANNELS'])), 'run_config.json of the previous run'
    return None, 'no record of the settings the masks were made with'

def reuse_decision():
    """(True/False, explanation): can the masks on disk be used for the current settings?"""
    if not REUSE_MASKS:
        return False, 'REUSE_MASKS = False'
    if not os.path.exists(MASK_FILE):
        return False, 'no saved masks'
    if WHOLE_CELL and not os.path.exists(CELL_MASK_FILE):
        return False, 'whole-cell masks are needed but were not made before'
    saved, source = saved_mask_settings()
    if saved is None:
        return False, source
    changed = [k for k in seg_settings if k in saved and saved[k] != seg_settings[k]]
    if changed:
        return False, 'changed since the last segmentation: ' + '; '.join(
            f'{k} {saved[k] or "none"} -> {seg_settings[k] or "none"}' for k in changed)
    return True, f'same settings (from {source})'

reuse, why = reuse_decision()
if reuse:
    nuc_raw = tifffile.imread(MASK_FILE).astype(np.int32)
    cells = tifffile.imread(CELL_MASK_FILE).astype(np.int32) if WHOLE_CELL else None
    print('loaded existing masks -', why)
    if not os.path.exists(MASK_INFO):                    # record the settings for the next run
        json.dump(dict(seg_settings, note='settings taken over from ' + why), open(MASK_INFO, 'w'), indent=1)
else:
    print('segmenting -', why)
    import spacec as sp
    seg_output = sp.tl.cell_segmentation(
        file_name=OUT_IMAGE, channel_file=CHANNEL_FILE, output_dir=output_dir,
        seg_method=SEG_METHOD, nuclei_channel=NUCLEUS_CHANNEL, output_fname=SAMPLE,
        membrane_channel_list=MEMBRANE_CHANNELS or None,
        compartment='whole-cell' if WHOLE_CELL else 'nuclei',
        differentiate_nucleus_cytoplasm=WHOLE_CELL,          # nuclear AND whole-cell masks
        input_format='Multichannel', resize_factor=RESIZE_FACTOR, size_cutoff=0)
    if WHOLE_CELL:
        nuc_raw = np.squeeze(np.asarray(seg_output['masks_nuclei'])).astype(np.int32)
        cells   = np.squeeze(np.asarray(seg_output['masks'])).astype(np.int32)
        tifffile.imwrite(CELL_MASK_FILE, cells, compression='zlib')
    else:
        nuc_raw = np.squeeze(np.asarray(seg_output['masks'])).astype(np.int32)
        cells = None
        if os.path.exists(CELL_MASK_FILE):               # whole-cell masks of an earlier setting are now invalid
            os.remove(CELL_MASK_FILE); print('removed old whole-cell masks (no membrane channels now)')
    del seg_output; gc.collect()
    tifffile.imwrite(MASK_FILE, nuc_raw, compression='zlib')
    json.dump(seg_settings, open(MASK_INFO, 'w'), indent=1)
    print('masks saved')
PARAM_LOG['segmentation'] = dict(seg_settings, reused=bool(reuse), reason=why)

if WHOLE_CELL:
    nuclei = np.where(nuc_raw > 0, cells, 0).astype(np.int32)   # nucleus carries the ID of its cell
else:
    nuclei = nuc_raw
del nuc_raw; gc.collect()
assert nuclei.shape == (H, W), f'mask shape {nuclei.shape} != image {(H, W)}'
print(f'{len(np.unique(nuclei)) - 1:,} segmented nuclei' + (' (whole-cell masks from ' + ', '.join(MEMBRANE_CHANNELS) + ')' if WHOLE_CELL else ''))


## 2.2 Cell compartments
Three compartments are available per cell: nucleus (`nuc`), whole cell (`cell`) and the ring around the nucleus (`ring`).
* With whole-cell segmentation, the cell outline comes from Mesmer.
* With nuclear segmentation, the nucleus can be expanded by `EXPAND_PX` pixels. The expansion is 0 by default; if `RING_MARKERS` are set, it is 3 px (about 1.5 µm), so that a ring exists to measure them in.

In [ ]:
EXPAND_PX = 3 if (RING_MARKERS and not WHOLE_CELL) else 0   # ring needed only for RING_MARKERS

def expand_labels_tiled(lab, d, tile=2048):
    """Grow labels by d px, processed in tiles to save memory."""
    if d <= 0:
        return lab
    out = np.zeros_like(lab); m = d + 2
    for y in range(0, lab.shape[0], tile):
        for x in range(0, lab.shape[1], tile):
            y0, x0 = max(y - m, 0), max(x - m, 0)
            ye, xe = min(y + tile, lab.shape[0]), min(x + tile, lab.shape[1])
            e = expand_labels(lab[y0:min(ye + m, lab.shape[0]), x0:min(xe + m, lab.shape[1])], d)
            out[y:ye, x:xe] = e[y - y0:ye - y0, x - x0:xe - x0]
    return out

if not WHOLE_CELL:
    cells = expand_labels_tiled(nuclei, EXPAND_PX)     # nuclear masks (+ optional ring)
NMAX = int(cells.max()) + 1

# QC: segmentation outlines on DAPI
y0, y1, x0, x1 = PREVIEW_CROP
y1, x1 = min(y1, y0 + 250), min(x1, x0 + 250)
d = read_plane(NUCLEUS_CHANNEL, OUT_IMAGE)[y0:y1, x0:x1]
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(d, cmap='gray', vmax=np.percentile(d, 99.5))
ax.contour(find_boundaries(nuclei[y0:y1, x0:x1]), [0.5], colors='cyan', linewidths=0.5)
if WHOLE_CELL or EXPAND_PX > 0:
    ax.contour(find_boundaries(cells[y0:y1, x0:x1]), [0.5], colors='yellow', linewidths=0.5)
ax.set_title('nuclei (cyan)' + (', cells (yellow)' if (WHOLE_CELL or EXPAND_PX) else '')); ax.axis('off')
finish('2-02_segmentation_crop', 'Segmentation (DAPI crop)', dpi=150)

## 2.3 Per-cell quantification
For every cell, the mean intensity of each channel is computed from the background-corrected image (used for the analysis) and from the raw image (used for comparisons only). All markers are measured in the **nucleus**, except `RING_MARKERS`, which are measured in the membrane/cytoplasm ring. This keeps the measurements identical across samples. Morphology, overlap with artefacts, distance to the tissue edge and whether the cell lies inside your GeoJSON outline (`in_region`) are recorded as well.

In [ ]:
COMPARTMENT = {m: 'nuc' for m in marker_channels}            # all markers in the nucleus
COMPARTMENT.update({m: 'ring' for m in RING_MARKERS})          # membrane markers in the ring

def label_sums(img, lab, chunk=1024):
    """Sum of pixel values per label (computed in row chunks)."""
    s = np.zeros(NMAX, np.float64)
    for y in range(0, lab.shape[0], chunk):
        s += np.bincount(lab[y:y + chunk].ravel(), weights=img[y:y + chunk].ravel(), minlength=NMAX)
    return s

n_nuc  = label_sums(np.ones((H, W), np.uint8), nuclei)      # pixels per nucleus
n_cell = label_sums(np.ones((H, W), np.uint8), cells)       # pixels per cell
n_ring = n_cell - n_nuc
labels = np.flatnonzero(n_nuc > 0); labels = labels[labels > 0]

def compartment_means(path):
    """Mean intensity per cell for every channel of an image, in the chosen compartment."""
    out = {}
    with tifffile.TiffFile(path) as tif:
        pages = _channel_pages(tif, len(channel_names))
        for name, page in zip(channel_names, pages):
            img = page.asarray()
            s_nuc, s_cell = label_sums(img, nuclei), label_sums(img, cells)
            with np.errstate(divide='ignore', invalid='ignore'):
                comp = dict(nuc=s_nuc / n_nuc, cell=s_cell / n_cell,
                            ring=np.where(n_ring > 0, (s_cell - s_nuc) / np.maximum(n_ring, 1), s_nuc / n_nuc))
            out[name] = comp[COMPARTMENT.get(name, 'nuc')][labels]
            del img
    return pd.DataFrame(out, index=pd.Index(labels, name='label'))

cor_df = compartment_means(OUT_IMAGE)   # background-corrected intensities
raw_all = compartment_means(RAW_IMAGE)  # raw intensities (comparison only)

# morphology per nucleus
props = ['label', 'centroid', 'area', 'eccentricity', 'axis_major_length', 'axis_minor_length', 'solidity']
try:
    morph = pd.DataFrame(regionprops_table(nuclei, properties=props))
except AttributeError:   # older scikit-image names
    props = ['label', 'centroid', 'area', 'eccentricity', 'major_axis_length', 'minor_axis_length', 'solidity']
    morph = pd.DataFrame(regionprops_table(nuclei, properties=props)).rename(
        columns={'major_axis_length': 'axis_major_length', 'minor_axis_length': 'axis_minor_length'})
morph = morph.rename(columns={'centroid-0': 'y', 'centroid-1': 'x'}).set_index('label').loc[labels]
morph['cell_area'] = n_cell[labels]
morph['artefact_frac'] = label_sums(artifact, cells)[labels] / n_cell[labels]   # overlap with artefacts
edge = ndi.distance_transform_edt(tissue_lr) * BG_DS                             # distance to tissue edge
morph['edge_dist'] = edge[np.clip((morph.y // BG_DS).astype(int), 0, edge.shape[0] - 1),
                          np.clip((morph.x // BG_DS).astype(int), 0, edge.shape[1] - 1)]

morph['in_region'] = points_in_geojson(REGION_GEOJSON, morph.x.to_numpy(), morph.y.to_numpy())

df_all = morph.join(raw_all[[NUCLEUS_CHANNEL]]).join(cor_df.drop(columns=NUCLEUS_CHANNEL))
print(f'{len(df_all):,} cells x {df_all.shape[1]} features')
df_all.head()

## 2.4 Cell quality filtering
Cells outside the GeoJSON outline are removed first. Of the remaining cells, objects that are unlikely to be reliable cells are removed:
* cells overlapping an artefact by more than `ARTEFACT_MAX_FRAC`;
* cells within `EDGE_EXCLUDE_PX` of the physical edge of the section (next to empty glass), where edge effects make staining unreliable. The border of your outline inside the tissue is not an edge, so cells along it are kept;
* objects that are too small or too large for a nucleus (by default below the 1st or above the 99.9th percentile of area, among the cells in the region);
* objects with too little DAPI signal (by default below the 1st percentile, among the cells in the region).

The red lines in the histograms mark the cut-offs, and the map shows where the removed cells are.

In [ ]:
ARTEFACT_MAX_FRAC = 0.05     # <-- EDIT
EDGE_EXCLUDE_PX   = 75       # <-- EDIT px (0 = keep edge cells)
AREA_MIN          = None     # <-- EDIT px²; None = 1st percentile
AREA_MAX          = None     # <-- EDIT px²; None = 99.9th percentile
DAPI_MIN          = None     # <-- EDIT; None = 1st percentile

base = df_all[df_all.in_region]                 # percentiles from the cells in the region
a_min = AREA_MIN if AREA_MIN is not None else np.percentile(base.area, 1)
a_max = AREA_MAX if AREA_MAX is not None else np.percentile(base.area, 99.9)
d_min = DAPI_MIN if DAPI_MIN is not None else np.percentile(base[NUCLEUS_CHANNEL], 1)
reasons = pd.DataFrame({
    'outside region': ~df_all.in_region,
    'artefact':    df_all.artefact_frac > ARTEFACT_MAX_FRAC,
    'tissue edge': df_all.edge_dist < EDGE_EXCLUDE_PX,
    'too small':   df_all.area < a_min,
    'too large':   df_all.area > a_max,
    'low DAPI':    df_all[NUCLEUS_CHANNEL] < d_min,
})
keep = ~reasons.any(axis=1)
print(f'area range [{a_min:.0f}, {a_max:.0f}] px², DAPI >= {d_min:.0f}')
print('removed per reason (a cell can fail several):'); print(reasons.sum().to_string())
print(f'kept {keep.sum():,} of {len(keep):,} cells ({100 * keep.mean():.1f} %)')

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(base.area, bins=100, range=(0, np.percentile(base.area, 99.95)), color='0.6')
for v in (a_min, a_max): ax[0].axvline(v, color='tab:red')
ax[0].set_title('nuclear area (px²)')
ax[1].hist(np.log10(base[NUCLEUS_CHANNEL] + 1), bins=100, color='0.6')
ax[1].axvline(np.log10(d_min + 1), color='tab:red'); ax[1].set_title('log10 DAPI intensity')
ax[2].scatter(df_all.x, df_all.y, s=0.2, c=np.where(keep, '0.8', 'red'), rasterized=True)
ax[2].invert_yaxis(); ax[2].set_aspect('equal'); ax[2].set_title('removed cells (red)'); ax[2].axis('off')
plt.tight_layout(); finish('2-04_cell_filtering', 'Cell quality filtering', dpi=150)

df_filtered = df_all[keep].copy()           # kept cells (re-running this cell is safe)
df = df_filtered.copy()
raw_df = raw_all.loc[df.index]
PARAM_LOG['cell_filters'] = dict(artefact_max_frac=ARTEFACT_MAX_FRAC, edge_px=EDGE_EXCLUDE_PX,
                                 area=[float(a_min), float(a_max)], dapi_min=float(d_min),
                                 expand_px=EXPAND_PX, compartment=COMPARTMENT,
                                 n_before=int(len(keep)), n_after=int(keep.sum()))

---
# Part 3: Cell-level background correction
Pixel-level correction removes the smooth background *between* cells. Two kinds of unwanted signal remain *inside* cells, and they are corrected per cell:

* **3.1 Residual spatial background**: in some regions, even negative cells read higher than elsewhere.
* **3.2 Non-specific signal**: some cells (e.g. sticky tumour cytoplasm, autofluorescent cells) are bright in *every* channel, including the non-functional antibodies.
* **3.3 Spillover** (optional): signal from a strongly stained neighbouring cell spilling into a cell.

## 3.1 Residual spatial background
For each marker, the slide is divided into bins of `CELL_BG_BIN` × `CELL_BG_BIN` px. In each bin, the `CELL_BG_Q`-th percentile of the cell values is taken as the local level of negative cells, smoothed across bins, and subtracted from every cell. Because a low percentile is used, positive cells keep their difference from the negative cells.

The table measures how uniform each marker is across the slide (0 = perfectly uniform), at three stages. The maps show the median per 500 px bin.

In [ ]:
CELL_BG           = True       # <-- EDIT
CELL_BG_BIN       = 400        # <-- EDIT bin size (px)
CELL_BG_Q         = 10         # <-- EDIT percentile used as local background
CELL_BG_SMOOTH    = 1.0        # smoothing (bins)
CELL_BG_MIN_CELLS = 30         # minimum cells per bin
CELL_BG_SKIP      = []         # <-- EDIT markers to leave unchanged

def spatial_floor(v, x, y, bin_px=CELL_BG_BIN, q=CELL_BG_Q):
    """Local low-percentile level of a marker at every cell position."""
    by, bx = (y // bin_px).astype(int).to_numpy(), (x // bin_px).astype(int).to_numpy()
    grid = np.full((by.max() + 1, bx.max() + 1), np.nan, np.float32)
    g = pd.DataFrame({'v': np.asarray(v), 'by': by, 'bx': bx}).groupby(['by', 'bx'])['v']
    qs, n = g.quantile(q / 100), g.size()
    ok = n[n >= CELL_BG_MIN_CELLS].index
    grid[ok.get_level_values(0), ok.get_level_values(1)] = qs.loc[ok].to_numpy()
    sm = fill_smooth(grid, CELL_BG_SMOOTH)
    return ndi.map_coordinates(sm, [y.to_numpy() / bin_px - 0.5, x.to_numpy() / bin_px - 0.5],
                               order=1, mode='nearest')

def bin_median_map(v, x, y, bin_px=TILE_QC_PX):
    """Median value per spatial bin (for the uniformity maps)."""
    t = pd.DataFrame({'v': np.asarray(v), 'ty': (y // bin_px).astype(int), 'tx': (x // bin_px).astype(int)})
    g = t.groupby(['ty', 'tx'])['v']
    return g.median().where(g.size() >= 20).unstack()

def spatial_nonuniformity(v, x, y):
    """SD of the bin medians relative to the marker's robust SD (0 = uniform)."""
    m = bin_median_map(v, x, y)
    return float(np.nanstd(m.to_numpy()) / max(robust_sd(v), 1e-9))

df = df_filtered.copy()                    # start from the filtered cells (re-running this cell is safe)
floor_markers = [m for m in PHENO_MARKERS + REF_CHANNELS if m not in CELL_BG_SKIP]
df_pix = df[PHENO_MARKERS + REF_CHANNELS].copy()          # values after pixel correction (for comparison)
if CELL_BG:
    for m in floor_markers:
        df[m] = df[m] - spatial_floor(df[m], df.x, df.y)
df_after_floor = df.copy()                  # input of 3.2

nonuni = pd.DataFrame({
    'raw':                    {m: spatial_nonuniformity(raw_df[m], df.x, df.y) for m in PHENO_MARKERS},
    'after pixel correction': {m: spatial_nonuniformity(df_pix[m], df.x, df.y) for m in PHENO_MARKERS},
    'after spatial floor':    {m: spatial_nonuniformity(df[m], df.x, df.y) for m in PHENO_MARKERS},
}).round(2)
print('spatial non-uniformity per marker (lower = more uniform):')
print(nonuni.to_string())

SHOW = [m for m in ['CD45', 'CD3E', 'CD68', 'col1A1'] if m in PHENO_MARKERS]
fig, axes = plt.subplots(len(SHOW), 3, figsize=(15, 3.6 * len(SHOW)), squeeze=False)
for r, m in enumerate(SHOW):
    for c, (src, ttl) in enumerate([(raw_df[m], 'raw'), (df_pix[m], 'after pixel correction'), (df[m], 'after spatial floor')]):
        mp = bin_median_map(src, df.x, df.y)
        s = robust_sd(src); mu = np.nanmedian(mp.to_numpy())
        im = axes[r, c].imshow((mp - mu) / s, cmap='RdBu_r', vmin=-1, vmax=1, interpolation='nearest')
        axes[r, c].set_title(f'{m}: {ttl}', fontsize=10); axes[r, c].axis('off')
    plt.colorbar(im, ax=axes[r, 2], fraction=0.03, label='bin median (robust SD units)')
plt.tight_layout(); finish('3-01_spatial_background', 'Residual spatial background per marker', dpi=150)

## 3.2 Non-specific signal: correction with the non-functional antibodies
The non-functional antibodies (`REF_CHANNELS`) do not bind any specific target. A cell that is bright in them is therefore bright because of non-specific binding or autofluorescence, and that signal is present in the working channels too.

1. Each reference channel is scaled to its 90th percentile. The **per-cell median across the reference channels** gives a `ref_score`, which measures how much non-specific signal the cell carries.
2. For each marker, cells are grouped into `REF_BINS` bins of `ref_score`. A robust (Theil–Sen) line is fitted through the median marker value per bin. The median follows the negative majority, so true positive cells barely influence the fit.
3. `slope × (ref_score − its 5th percentile)` is subtracted from every cell.

**Plots:** before correction, the cloud of negative cells rises with `ref_score` (red line = fit). After correction it is flat, and the true positives stand out above it. The table gives the correlation of each marker with `ref_score`, which should drop to about 0.

In [ ]:
REF_CORRECT = True          # <-- EDIT
REF_BINS    = 20            # number of ref_score bins for the fit
REF_FIT_Q   = 50            # percentile per bin the line is fitted to
REF_SKIP    = []            # <-- EDIT markers to leave unchanged

df = df_after_floor.copy()                 # start from the 3.1 result (re-running this cell is safe)
ref_scaled = df[REF_CHANNELS] / df[REF_CHANNELS].quantile(0.9).clip(lower=1e-6)
ref_score = ref_scaled.median(axis=1)                    # non-specific signal per cell
ref0 = ref_score.quantile(0.05)
ref_bins = pd.qcut(ref_score.rank(method='first'), REF_BINS, labels=False)
xb = ref_score.groupby(ref_bins).median()

ref_fit, df_before_ref = {}, df[PHENO_MARKERS].copy()
for m in PHENO_MARKERS:
    yb = df[m].groupby(ref_bins).quantile(REF_FIT_Q / 100)
    slope, intercept, lo, hi = theilslopes(yb.to_numpy(), xb.to_numpy())    # robust line fit
    beta = max(float(slope), 0.0) if (REF_CORRECT and m not in REF_SKIP) else 0.0
    ref_fit[m] = dict(slope=beta, intercept=float(intercept))
    df[m] = df[m] - beta * (ref_score - ref0)                               # remove non-specific part
df['ref_score'] = ref_score
df_after_ref = df.copy()                    # input of 3.3

corr = pd.DataFrame({
    'rho raw':        {m: spearmanr(raw_df[m], ref_score)[0] for m in PHENO_MARKERS},
    'rho before 3.2': {m: spearmanr(df_before_ref[m], ref_score)[0] for m in PHENO_MARKERS},
    'rho after 3.2':  {m: spearmanr(df[m], ref_score)[0] for m in PHENO_MARKERS},
    'slope':          {m: ref_fit[m]['slope'] for m in PHENO_MARKERS},
}).round(2)
print('Spearman correlation of each marker with the non-specific signal (should drop to ~0):')
print(corr.to_string())
PARAM_LOG['ref_correction'] = dict(channels=REF_CHANNELS, fit=ref_fit, ref0=float(ref0))

show = LINEAGE_MARKERS[:12]
NC = 6; NR = int(np.ceil(len(show) / NC))
fig, axes = plt.subplots(2 * NR, NC, figsize=(2.9 * NC, 2.5 * 2 * NR), squeeze=False)
xs = np.clip(ref_score, 0, ref_score.quantile(0.995))
for i, m in enumerate(show):
    for k, (v, ttl) in enumerate([(df_before_ref[m], 'before'), (df[m], 'after')]):
        ax = axes[2 * (i // NC) + k, i % NC]
        hi = np.nanpercentile(v, 99.5); lo = np.nanpercentile(v, 0.5)
        ax.hexbin(xs, np.clip(v, lo, hi), gridsize=40, bins='log', cmap='viridis', mincnt=1)
        if k == 0:
            ax.plot(xb, df_before_ref[m].groupby(ref_bins).quantile(REF_FIT_Q / 100), 'w.', ms=4)
            ax.plot(xb, ref_fit[m]['intercept'] + ref_fit[m]['slope'] * xb, 'r-', lw=1)
        ax.set_title(f'{m} {ttl}', fontsize=8); ax.tick_params(labelsize=6)
for ax in axes.ravel()[2 * NR * NC:]:
    ax.axis('off')
fig.supxlabel('ref_score (non-specific signal)')
plt.tight_layout(); finish('3-02_nonspecific_signal', 'Non-specific signal correction', dpi=150)

## 3.3 Spillover compensation (optional, off)
A REDSEA-like correction for signal spilling over from neighbouring cells. For each cell, `ALPHA` × (shared-border fraction × neighbour value) is subtracted, summed over all touching neighbours. Turn it on (`COMPENSATE = True`) if clusters in Part 4 are positive for two mutually exclusive lineages at the same time, e.g. CD3E and CD68, at the border of dense immune regions. It was not used for this dataset.

In [ ]:
COMPENSATE = False     # <-- EDIT
ALPHA      = 0.3       # <-- EDIT strength (0.2 - 0.5)

def contact_matrix(lab, index_labels, chunk=2048):
    """Fraction of each cell's border shared with each neighbouring cell (sparse matrix)."""
    import scipy.sparse as sps
    lut = np.full(NMAX, -1, np.int64); lut[index_labels] = np.arange(len(index_labels))
    perim = np.zeros(NMAX, np.float64); pairs = []
    for y in range(0, lab.shape[0], chunk):
        blk = lab[y:min(y + chunk + 1, lab.shape[0])]
        for a, b in ((blk[:chunk, :-1], blk[:chunk, 1:]), (blk[:-1], blk[1:])):
            d = a != b
            perim += np.bincount(a[d & (a > 0)], minlength=NMAX) + np.bincount(b[d & (b > 0)], minlength=NMAX)
            both = d & (a > 0) & (b > 0)
            pairs.append(np.stack([a[both], b[both]], 1))
    p = np.concatenate(pairs).astype(np.int64)
    i, j = lut[p[:, 0]], lut[p[:, 1]]; ok = (i >= 0) & (j >= 0)
    n = len(index_labels)
    C = sps.coo_matrix((np.ones(ok.sum()), (i[ok], j[ok])), shape=(n, n)).tocsr()
    C = C + C.T
    P = perim[index_labels]
    return sps.diags(1.0 / np.maximum(P, 1)) @ C

df = df_after_ref.copy()                    # start from the 3.2 result (re-running this cell is safe)
if COMPENSATE:
    Wn = contact_matrix(cells, df.index.to_numpy())
    comp_markers = [m for m in PHENO_MARKERS if COMPARTMENT.get(m) != 'nuc']
    X = df[comp_markers].clip(lower=0).to_numpy()
    df[comp_markers] = df[comp_markers].to_numpy() - ALPHA * (Wn @ X)
    print(f'compensated {len(comp_markers)} markers with alpha={ALPHA}')
else:
    print('spillover compensation: off')
PARAM_LOG['spillover'] = dict(compensate=COMPENSATE, alpha=ALPHA)

## 3.4 Save the results of this notebook
* **`<sample>_cells_corrected.csv`**: the corrected per-cell table, which is the input for 03b.
* `<sample>_run_config.json`: sample settings (panel, image size), read by 03b.
* `<sample>_masks_lowres.npz`: tissue and analysis-region masks (down-sampled), used by 03b for the region area.
* `<sample>_parameters.json`: all preprocessing settings, continued by 03b.

In [ ]:
df.to_csv(CORRECTED_CSV)
np.savez_compressed(LOWRES_MASKS, tissue_lr=tissue_lr, region_lr=region_lr, art_lr=art_lr, bg_ds=BG_DS)
run_config = dict(SAMPLE=SAMPLE, CONDITION=CONDITION, REGION=REGION, PLOT_TAG=PLOT_TAG, H=int(H), W=int(W),
                  BG_DS=BG_DS, NUCLEUS_CHANNEL=NUCLEUS_CHANNEL, channel_names=channel_names,
                  marker_channels=marker_channels, LINEAGE_MARKERS=LINEAGE_MARKERS, STATE_MARKERS=STATE_MARKERS,
                  NUCLEAR_MARKERS=NUCLEAR_MARKERS, NONFUNCTIONAL=NONFUNCTIONAL, REF_CHANNELS=REF_CHANNELS,
                  FAILED_MARKERS=FAILED_MARKERS,
                  MEMBRANE_CHANNELS=MEMBRANE_CHANNELS, RING_MARKERS=RING_MARKERS, REGION_GEOJSON=REGION_GEOJSON,
                  n_cells=int(len(df)))
json.dump(run_config, open(RUN_CONFIG, 'w'), indent=1)

def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o
json.dump(_jsonable(PARAM_LOG), open(OUT + 'parameters.json', 'w'), indent=1)
print('written:', CORRECTED_CSV, df.shape)
print('done:', PLOT_TAG)